In [1]:
# =========================================================
# 04_dziribert_train.ipynb
# Purpose:
#   - Load the final cleaned sentiment dataset
#   - Split into train / validation / test
#   - Tokenize text using DziriBERT tokenizer
#   - Fine-tune DziriBERT for 3-class sentiment analysis
#   - Evaluate the model
#   - Save the final trained model and tokenizer
#
# Final label convention:
#   0 = negative
#   1 = neutral
#   2 = positive
# =========================================================

In [2]:
# ---------------------------------------------------------
# Standard library
# ---------------------------------------------------------
from pathlib import Path
import json
import os

# ---------------------------------------------------------
# Data analysis
# ---------------------------------------------------------
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Machine learning / evaluation
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ---------------------------------------------------------
# PyTorch
# ---------------------------------------------------------
import torch

# ---------------------------------------------------------
# Hugging Face
# ---------------------------------------------------------
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# ---------------------------------------------------------
# Define the workspace paths
# ---------------------------------------------------------

ML_ROOT = Path.cwd().parent

FINAL_DATA_DIR = ML_ROOT / "data" / "final"
ARTIFACTS_DIR = ML_ROOT / "artifacts"

# Final cleaned dataset produced by notebook 02
final_dataset_path = FINAL_DATA_DIR / "sentiment_dataset_final.csv"

# Where we will save the final model
MODEL_OUTPUT_DIR = ARTIFACTS_DIR / "dziribert_sentiment_model"

# Create output directory if it does not exist
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Final dataset path:", final_dataset_path)
print("Model output dir:", MODEL_OUTPUT_DIR)

Final dataset path: C:\Users\hp\Lucidia-project\ml\data\final\sentiment_dataset_final.csv
Model output dir: C:\Users\hp\Lucidia-project\ml\artifacts\dziribert_sentiment_model


In [4]:
# ---------------------------------------------------------
# Check if GPU is available
# ---------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

if device == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training will run on CPU and may be slow.")

Using device: cpu
No GPU detected. Training will run on CPU and may be slow.


In [5]:
# ---------------------------------------------------------
# Load the final cleaned dataset
# ---------------------------------------------------------

df = pd.read_csv(final_dataset_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (11492, 5)


,text,label,label_raw,source_dataset,text_length_chars
0,استفيدوا من عروض جازي عايلة الجديدة لي توالم ك...,1,0.0,dataset_1,340
1,Conx ta3koum dayra ki lhaaaam w say,0,-1.0,dataset_1,35
2,@zaki_medjber_62,1,0.0,dataset_1,16
3,ما فهمتش هي sim وحدة يخدمو بها العائلة كاملة,1,0.0,dataset_1,44
4,🙏😉🙏,2,1.0,dataset_1,3


In [6]:
# ---------------------------------------------------------
# Basic sanity checks
# ---------------------------------------------------------

print("Columns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

print("\nMissing values:")
print(df.isna().sum())

Columns: ['text', 'label', 'label_raw', 'source_dataset', 'text_length_chars']

Label distribution:
label
0    4687
1    4971
2    1834
Name: count, dtype: int64

Missing values:
text                 0
label                0
label_raw            0
source_dataset       0
text_length_chars    0
dtype: int64


In [7]:
# ---------------------------------------------------------
# Keep only the essential columns for model training
# ---------------------------------------------------------

df = df[["text", "label"]].copy()

# Ensure text is string and label is integer
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

display(df.head())
print(df.dtypes)

,text,label
0,استفيدوا من عروض جازي عايلة الجديدة لي توالم ك...,1
1,Conx ta3koum dayra ki lhaaaam w say,0
2,@zaki_medjber_62,1
3,ما فهمتش هي sim وحدة يخدمو بها العائلة كاملة,1
4,🙏😉🙏,2


text       str
label    int64
dtype: object


In [8]:
# ---------------------------------------------------------
# Define human-readable label mappings
# These mappings will be saved with the model later
# ---------------------------------------------------------

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

print("id2label:", id2label)
print("label2id:", label2id)

id2label: {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id: {'negative': 0, 'neutral': 1, 'positive': 2}


In [9]:
# ---------------------------------------------------------
# Split the data into:
#   - 70% train
#   - 15% validation
#   - 15% test
#
# We use stratify=y to preserve class distribution across splits
# ---------------------------------------------------------

X = df["text"]
y = df["label"]

# First split: train vs temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

# Second split: validation vs test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

Train size: 8044
Validation size: 1724
Test size: 1724


In [10]:
# ---------------------------------------------------------
# Convert splits into DataFrames
# This makes it easy to convert them into Hugging Face Datasets
# ---------------------------------------------------------

train_df = pd.DataFrame({"text": X_train, "label": y_train}).reset_index(drop=True)
val_df = pd.DataFrame({"text": X_val, "label": y_val}).reset_index(drop=True)
test_df = pd.DataFrame({"text": X_test, "label": y_test}).reset_index(drop=True)

print("Train DataFrame shape:", train_df.shape)
print("Validation DataFrame shape:", val_df.shape)
print("Test DataFrame shape:", test_df.shape)

display(train_df.head())

Train DataFrame shape: (8044, 2)
Validation DataFrame shape: (1724, 2)
Test DataFrame shape: (1724, 2)


,text,label
0,@souad_otm كيف كيف,1
1,نسرين زيدي ريبوندلي باش نربح كما المرة لي فاتت...,1
2,Très mauvaise connexion 😤😤😤,0
3,Hadra illimité avec djezzy khier m'en Youtube,0
4,كونكسيون ميتة,0


In [11]:
# ---------------------------------------------------------
# Convert pandas DataFrames to Hugging Face Dataset format
# ---------------------------------------------------------

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8044
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1724
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1724
    })
})

In [12]:
# ---------------------------------------------------------
# Load DziriBERT tokenizer
#
# This is the tokenizer that matches the pretrained model
# If this is the first time you run it, it may download files
# from Hugging Face
# ---------------------------------------------------------

MODEL_NAME = "alger-ia/dziribert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [13]:
# ---------------------------------------------------------
# Set tokenization parameters
#
# The thesis reported best performance with max_length = 65
# so we start with that exact value
# ---------------------------------------------------------

MAX_LENGTH = 65

print("Using max_length =", MAX_LENGTH)

Using max_length = 65


In [14]:
# ---------------------------------------------------------
# Define tokenization function
#
# We truncate long texts to MAX_LENGTH
# Padding will be handled dynamically later by the data collator
# ---------------------------------------------------------

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

In [15]:
# ---------------------------------------------------------
# Tokenize all dataset splits
# ---------------------------------------------------------

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)

tokenized_datasets

Map:   0%|          | 0/8044 [00:00<?, ? examples/s]

Map:   0%|          | 0/1724 [00:00<?, ? examples/s]

Map:   0%|          | 0/1724 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8044
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1724
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1724
    })
})

In [16]:
# ---------------------------------------------------------
# Keep only the columns needed by the Trainer
# ---------------------------------------------------------

tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# Ensure labels are named exactly "label"
# Hugging Face Trainer expects this by default
tokenized_datasets.set_format("torch")

tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8044
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1724
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1724
    })
})

In [17]:
# ---------------------------------------------------------
# Data collator dynamically pads each batch
# This is more efficient than padding the whole dataset to max length
# ---------------------------------------------------------

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Data collator ready.")

Data collator ready.


In [18]:
# ---------------------------------------------------------
# Load pretrained DziriBERT and add a classification head
# for 3 sentiment classes
# ---------------------------------------------------------
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.2,  # thesis-guided value
    attention_probs_dropout_prob=0.2,  # keep consistent with dropout idea
)

model.to(device)

print("Model loaded successfully.")

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

C:\Users\hp\Lucidia-project\ml\clean_venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--alger-ia--dziribert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on

Model loaded successfully.


In [19]:
# ---------------------------------------------------------
# Define evaluation metrics
#
# We track:
#   - accuracy
#   - precision
#   - recall
#   - f1
#
# We use macro averaging because the dataset is imbalanced
# and we want balanced performance across classes
# ---------------------------------------------------------

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [25]:
training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR / "checkpoints"),

    # Core training setup
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # Evaluation / saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Logging (safe)
    logging_strategy="epoch",

    # Reproducibility
    seed=42,
    report_to="none",

    # Regularization
    weight_decay=0.01,
)

In [26]:
# ---------------------------------------------------------
# Create the Hugging Face Trainer
#
# The Trainer will handle:
#   - training loop
#   - validation at each epoch
#   - metric computation
#   - saving the best model
#
# It uses:
#   - the model
#   - training arguments
#   - tokenized train dataset
#   - tokenized validation dataset
#   - tokenizer
#   - data collator for dynamic padding
#   - custom metrics function
# ---------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer created successfully.")

Trainer created successfully.


In [27]:
# ---------------------------------------------------------
# Start fine-tuning DziriBERT
#
# This is the main training step.
# The model will:
#   - train on the training set
#   - evaluate on the validation set every epoch
#   - keep track of the best validation performance
#
# IMPORTANT:
# Since you are on CPU, training may take some time.
# Be patient and let the process finish.
# ---------------------------------------------------------

train_result = trainer.train()

print("Training completed successfully.")

C:\Users\hp\Lucidia-project\ml\clean_venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.774933,0.718865,0.668213,0.668169,0.694288,0.678878
2,0.551508,0.741036,0.688515,0.711156,0.703094,0.706664
3,0.409015,0.886604,0.694316,0.702510,0.714751,0.708251


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\hp\Lucidia-project\ml\clean_venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\hp\Lucidia-project\ml\clean_venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training completed successfully.


In [28]:
# ---------------------------------------------------------
# Display the final training output summary
# This usually contains:
#   - training runtime
#   - samples per second
#   - training loss
# ---------------------------------------------------------

print("Training result summary:")
print(train_result)

Training result summary:
TrainOutput(global_step=3018, training_loss=0.5784850831534232, metrics={'train_runtime': 11922.6263, 'train_samples_per_second': 2.024, 'train_steps_per_second': 0.253, 'total_flos': 469051056439560.0, 'train_loss': 0.5784850831534232, 'epoch': 3.0})


In [30]:
# ---------------------------------------------------------
# Evaluate the model on the validation set
#
# This gives us the performance of the best loaded model
# on unseen validation data.
#
# Metrics returned should include:
#   - eval_loss
#   - eval_accuracy
#   - eval_precision_macro
#   - eval_recall_macro
#   - eval_f1_macro
# ---------------------------------------------------------

# Safe validation eval (avoids callback issue)
val_results = trainer.predict(tokenized_datasets["validation"])
val_metrics = trainer.compute_metrics(val_results)
print("Validation metrics:", {k: f"{v:.4f}" for k, v in val_metrics.items()})

# Safe test eval and predictions
test_results = trainer.predict(tokenized_datasets["test"])
test_metrics = trainer.compute_metrics(test_results)
test_logits = test_results.predictions
test_labels = test_results.label_ids
test_preds = np.argmax(test_logits, axis=1)

print("Test metrics:", {k: f"{v:.4f}" for k, v in test_metrics.items()})
print("\nClassification Report:\n", classification_report(test_labels, test_preds, 
    target_names=["negative", "neutral", "positive"], digits=4, zero_division=0))


C:\Users\hp\Lucidia-project\ml\clean_venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


ValueError: too many values to unpack (expected 2)

In [ ]:
# ---------------------------------------------------------
# Evaluate on the held-out test set
#
# This is the most important final performance estimate.
# The test set was never used for training or validation.
# ---------------------------------------------------------

test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("Test metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {value}")

In [ ]:
# ---------------------------------------------------------
# Generate predictions on the test set
#
# We will use these predictions for:
#   - classification report
#   - confusion matrix
#   - manual error analysis
# ---------------------------------------------------------

pred_output = trainer.predict(tokenized_datasets["test"])

test_logits = pred_output.predictions
test_labels = pred_output.label_ids
test_preds = np.argmax(test_logits, axis=1)

print("Predictions generated successfully.")
print("Number of test examples:", len(test_preds))

In [ ]:
# ---------------------------------------------------------
# Print detailed classification report
#
# This report shows performance for each class:
#   - precision
#   - recall
#   - f1-score
#   - support
#
# Class order:
#   0 = negative
#   1 = neutral
#   2 = positive
# ---------------------------------------------------------

print("Classification Report on Test Set:\n")

report = classification_report(
    test_labels,
    test_preds,
    target_names=["negative", "neutral", "positive"],
    digits=4,
    zero_division=0,
)

print(report)

In [ ]:
# ---------------------------------------------------------
# Plot confusion matrix
#
# This helps us understand:
#   - which classes are predicted well
#   - where the model gets confused
#
# In this task, confusion often appears between:
#   - neutral vs negative
#   - neutral vs positive
# ---------------------------------------------------------

cm = confusion_matrix(test_labels, test_preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["negative", "neutral", "positive"],
)

disp.plot(cmap="Blues")
plt.title("DziriBERT Confusion Matrix - Test Set")
plt.show()

In [ ]:
# ---------------------------------------------------------
# Create a DataFrame with:
#   - original text
#   - true label
#   - predicted label
#   - readable label names
#   - correctness flag
#
# This will help us inspect examples manually.
# ---------------------------------------------------------

test_results_df = test_df.copy()

test_results_df["true_label"] = test_labels
test_results_df["pred_label"] = test_preds

test_results_df["true_label_name"] = test_results_df["true_label"].map(id2label)
test_results_df["pred_label_name"] = test_results_df["pred_label"].map(id2label)

test_results_df["is_correct"] = test_results_df["true_label"] == test_results_df["pred_label"]

display(test_results_df.head(10))

In [ ]:
# ---------------------------------------------------------
# Filter only incorrect predictions
#
# These are the most useful examples for error analysis.
# By reading them, we can understand where the model struggles.
# ---------------------------------------------------------

errors_df = test_results_df[test_results_df["is_correct"] == False].copy()

print("Number of incorrect predictions:", len(errors_df))
display(errors_df.head(20))

In [ ]:
# ---------------------------------------------------------
# Show some correct predictions
#
# This is useful to compare what the model handles well
# versus what it misclassifies.
# ---------------------------------------------------------

correct_df = test_results_df[test_results_df["is_correct"] == True].copy()

print("Number of correct predictions:", len(correct_df))
display(correct_df.head(20))

In [ ]:
# ---------------------------------------------------------
# Save the final trained model and tokenizer
#
# These files will later be loaded in Django for inference.
# ---------------------------------------------------------

FINAL_MODEL_DIR = MODEL_OUTPUT_DIR / "final_model"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

print("Final model and tokenizer saved to:")
print(FINAL_MODEL_DIR)

In [ ]:
# ---------------------------------------------------------
# Save the label mappings
#
# This will help us decode predictions later in the API
# without hardcoding labels manually.
# ---------------------------------------------------------

label_map_path = FINAL_MODEL_DIR / "label_map.json"

with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "id2label": {str(k): v for k, v in id2label.items()},
            "label2id": label2id,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Label map saved to:")
print(label_map_path)

In [ ]:
# ---------------------------------------------------------
# Save all test predictions to CSV
#
# This file is useful for:
#   - manual review
#   - debugging
#   - comparing future model versions
# ---------------------------------------------------------

predictions_output_path = FINAL_MODEL_DIR / "test_predictions.csv"

test_results_df.to_csv(predictions_output_path, index=False, encoding="utf-8-sig")

print("Test predictions saved to:")
print(predictions_output_path)

In [ ]:
# ---------------------------------------------------------
# Save validation and test metrics in JSON format
#
# This makes it easy to compare model runs later.
# ---------------------------------------------------------

metrics_output_path = FINAL_MODEL_DIR / "metrics.json"

metrics_to_save = {
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
}

with open(metrics_output_path, "w", encoding="utf-8") as f:
    json.dump(metrics_to_save, f, indent=2)

print("Metrics saved to:")
print(metrics_output_path)

In [ ]:
# ---------------------------------------------------------
# Define a simple helper function for manual prediction
#
# This allows us to test the model on custom text directly
# inside the notebook after training.
# ---------------------------------------------------------

def predict_sentiment(text: str):
    """
    Predict sentiment for one input text.
    Returns:
        - predicted label id
        - predicted label name
        - class probabilities
    """
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    # Move tensors to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]

    return {
        "text": text,
        "predicted_label_id": pred_id,
        "predicted_label": pred_label,
        "probabilities": {
            "negative": float(probs[0]),
            "neutral": float(probs[1]),
            "positive": float(probs[2]),
        },
    }

In [ ]:
# ---------------------------------------------------------
# Test the model on custom examples
# You can replace these with real Algerian customer comments
# ---------------------------------------------------------

example_1 = "الخدمة كارثية اليوم وماكان حتى ريزو"
example_2 = "التطبيق مليح بزاف وسهل الاستعمال"
example_3 = "عادي، ماشي مليح وماشي خايب"

print(predict_sentiment(example_1))
print(predict_sentiment(example_2))
print(predict_sentiment(example_3))

In [ ]:
# ---------------------------------------------------------
# Final notebook summary
# ---------------------------------------------------------

print("Notebook 04 completed.")
print("- DziriBERT was fine-tuned on the cleaned sentiment dataset.")
print("- Validation and test evaluation were completed.")
print("- Classification report and confusion matrix were generated.")
print("- Final model, tokenizer, label map, predictions, and metrics were saved.")
print("- Ready for final model review and Django integration.")